# Lab 2: Simulation and Bayesian Updating

**Phân tích dữ liệu Bayesian - IUH**

Một game show với nhiều cánh cửa, một chuỗi xét nghiệm COVID, một bài toán bốc bi hay một ván bài quá nhiều trường hợp để đếm tay. Từ lab này trở đi, xác suất không còn chỉ nằm trên giấy. Ta bắt đầu dùng mô phỏng như một cách nhìn trực tiếp vào cơ chế sinh dữ liệu.

Điểm đáng học ở đây là mô phỏng không thay thế lập luận xác suất. Ngược lại, nó buộc ta mô tả cơ chế của bài toán cho thật đúng. Nếu mô tả sai cách dữ liệu được tạo ra, mô phỏng càng chạy nhiều chỉ càng xác nhận một trực giác sai.


In [ ]:
# Dùng cả công thức lẫn mô phỏng để sinh viên thấy hai cách tiếp cận kiểm chứng lẫn nhau.
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb

rng = np.random.default_rng(2024)
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Từ công thức sang mô phỏng

Lab này xoay quanh những bài toán mà cơ chế sinh dữ liệu giữ vai trò quyết định. Trong Monty Hall, hành vi của người dẫn chương trình là một phần của mô hình. Trong bài bốc bi, sự khác nhau giữa “có hoàn lại” và “không hoàn lại” làm đổi hẳn cấu trúc xác suất. Trong chuỗi xét nghiệm COVID, niềm tin không được cập nhật một lần, mà dịch chuyển sau từng kết quả mới.

Chính vì thế, mô phỏng xuất hiện như một người bạn đồng hành của công thức. Nó không làm bài toán dễ đi bằng mẹo, mà giúp ta kiểm tra xem mình đã mô tả đúng câu chuyện xác suất hay chưa. Khi công thức và mô phỏng cùng chỉ về một kết quả, trực giác của ta trở nên vững hơn rất nhiều.


## 2. Task 1 - Monty Hall với 3 cửa và với $n$ cửa

Người dẫn chương trình không mở cửa ngẫu nhiên mà mở có điều kiện trên vị trí xe. Chính điều kiện đó làm posterior nghiêng về phương án đổi cửa.

### Ý nghĩa của bài tập

Monty Hall là bài tập hoàn hảo để thấy dữ liệu chỉ có ý nghĩa khi gắn với cơ chế sinh dữ liệu. Người dẫn chương trình không mở cửa ngẫu nhiên; ông ta hành động có điều kiện theo vị trí xe. Chính sự phụ thuộc đó tạo ra lợi thế của chiến lược đổi cửa.

Bài này cũng cho thấy simulation hữu ích nhất khi nó tôn trọng đúng cơ chế của bài toán. Nếu mô phỏng sai cách người dẫn chọn cửa, ta sẽ học sai trực giác dù chạy hàng triệu lần.


### Đề bài nhắc lại

**Monty Hall problem.** Có 3 cửa chứa 1 xe và 2 con dê. Người chơi chọn một cửa, người dẫn biết vị trí xe nên chỉ mở một cửa có dê, rồi cho người chơi quyền đổi sang cửa còn lại. Hãy phân tích xác suất thắng nếu giữ nguyên hay đổi cửa. Sau đó mở rộng bài toán khi số cánh cửa là $n$ bất kỳ.


In [ ]:
# Mỗi lượt chơi Monty Hall được mô phỏng đúng cơ chế: người dẫn chỉ mở cửa dê và để lại đúng một lựa chọn đổi cửa.
def monty_hall_trial(n_doors=3):
    doors = list(range(n_doors))
    car = rng.integers(n_doors)
    choice = rng.integers(n_doors)
    remaining = [d for d in doors if d not in {choice, car}]
    host_opens = set(rng.choice(remaining, size=n_doors-2, replace=False))
    switch_option = next(d for d in doors if d not in host_opens and d != choice)
    return choice == car, switch_option == car

stay_prob_theory = 1/3
switch_prob_theory = 2/3
sim = np.array([monty_hall_trial(3) for _ in range(20000)])
print(f"3-door stay: theory={stay_prob_theory:.4f}, simulation={sim[:,0].mean():.4f}")
print(f"3-door switch: theory={switch_prob_theory:.4f}, simulation={sim[:,1].mean():.4f}")

n_values = np.arange(3, 11)
switch_theory = (n_values - 1) / n_values
switch_sim = []
for n in n_values:
    result = np.array([monty_hall_trial(n) for _ in range(5000)])
    switch_sim.append(result[:, 1].mean())

plt.plot(n_values, switch_theory, marker='o', label='theory')
plt.plot(n_values, switch_sim, marker='s', label='simulation')
plt.xlabel('number of doors')
plt.ylabel('P(win by switching)')
plt.title('Monty Hall tổng quát')
plt.legend()
plt.show()


### Interpretation

Với 3 cửa, xác suất thắng khi đổi là $\frac{2}{3}$. Với $n$ cửa, xác suất này trở thành $\frac{n-1}{n}$ vì xác suất chọn sai ngay từ đầu là rất lớn, còn người dẫn lại dồn toàn bộ xác suất đó vào một cửa còn lại. Simulation chỉ đóng vai trò xác nhận kết quả lý thuyết.


## 3. Task 2 - Bốc 10 viên bi và xác suất đúng 5 đỏ

Ta so sánh hai cơ chế sinh dữ liệu khác nhau: có hoàn lại và không hoàn lại. Dù cùng hỏi “đúng 5 bi đỏ”, hai mô hình xác suất khác nhau vì sự phụ thuộc giữa các lượt bốc khác nhau.

### Ý nghĩa của bài tập

Bài này dạy một phân biệt then chốt trong xác suất: cùng một câu hỏi đếm có thể cho hai đáp án khác nhau chỉ vì cơ chế lấy mẫu khác nhau. “Có hoàn lại” tạo ra các lượt bốc độc lập, còn “không hoàn lại” tạo ra phụ thuộc âm giữa các lượt.

Ý nghĩa sâu hơn là ta không bao giờ nên tách bài toán khỏi câu chuyện sinh dữ liệu. Công thức đúng không phải công thức quen tay nhất, mà là công thức phù hợp với cách dữ liệu thực sự được tạo ra.


### Đề bài nhắc lại

**Bài toán bốc bi.** Trong hộp có $8$ bi đỏ và $12$ bi đen. Bốc ngẫu nhiên $10$ viên. Tính xác suất bốc được đúng $5$ bi đỏ trong hai trường hợp: có hoàn lại và không hoàn lại.


In [ ]:
# So sánh cùng một câu hỏi “đúng 5 bi đỏ” dưới hai cơ chế sinh dữ liệu khác nhau: có hoàn lại và không hoàn lại.
exact_with_replacement = comb(10, 5) * (8/20)**5 * (12/20)**5
exact_without_replacement = comb(8, 5) * comb(12, 5) / comb(20, 10)

print(f"With replacement: {exact_with_replacement:.6f}")
print(f"Without replacement: {exact_without_replacement:.6f}")

def draw_with_replacement():
    draws = rng.integers(0, 20, size=10)
    return np.sum(draws < 8) == 5

def draw_without_replacement():
    draws = rng.choice(np.arange(20), size=10, replace=False)
    return np.sum(draws < 8) == 5

N = 30000
sim_wr = np.mean([draw_with_replacement() for _ in range(N)])
sim_wor = np.mean([draw_without_replacement() for _ in range(N)])
print(f"Simulation with replacement: {sim_wr:.6f}")
print(f"Simulation without replacement: {sim_wor:.6f}")


## 4. Task 3 - TOEIC improvement và sản phẩm hỏng ở lượt thứ 10

Task này gom hai bài nhỏ nhưng rất có giá trị vì chúng rèn cho người học thói quen nhận ra cấu trúc đơn giản ẩn sau một đề bài có vẻ dài dòng. Bài TOEIC thực chất là bài đếm có điều kiện trên một không gian điểm số rời rạc, còn bài sản phẩm hỏng ở lượt thứ 10 lại là bài đối xứng vị trí.

### Ý nghĩa của bài tập

Ý nghĩa của cặp bài này là nhắc ta không phải lúc nào bài toán xác suất cũng cần machinery nặng. Nhiều khi điều quan trọng nhất là nhận ra đâu là không gian mẫu, đâu là điều kiện đang áp dụng, và khi nào có thể dùng đối xứng để rút gọn lập luận.

Đây cũng là kỹ năng rất cần trong Bayesian analysis: trước khi viết posterior, ta phải nhìn ra cấu trúc của dữ liệu và không lạm dụng mô hình phức tạp khi một lập luận cơ bản đã đủ mạnh.


### Đề bài nhắc lại

Hai bài nhỏ ở phần này là:

- **TOEIC improvement.** Điểm TOEIC là bội số của $5$ từ $0$ đến $990$. Sinh viên thi lần 1 được $x$ điểm, biết rằng $x<500$, rồi thi cải thiện lần 2 được $y$ điểm. Tính xác suất để sinh viên hài lòng, tức là $y>x$.
- **Sản phẩm hỏng ở lượt thứ 10.** Có $1000$ sản phẩm trong đó có $10$ cái hỏng. Một người bốc lần lượt $10$ lần, mỗi lần một sản phẩm. Tính xác suất để lần thứ $10$ bốc phải sản phẩm hỏng.


In [ ]:
# Hai bài nhỏ này nhấn mạnh sức mạnh của đếm trực tiếp và đối xứng trước khi cần đến mô phỏng nặng hơn.
scores = list(range(0, 995, 5))
first_scores = [x for x in scores if x < 500]
prob_toeic = np.mean([np.mean(np.array(scores) > x) for x in first_scores])
prob_defective_10th = 10 / 1000

print(f"P(y > x | x < 500) = {prob_toeic:.6f}")
print(f"P(10th item is defective) = {prob_defective_10th:.6f}")


### Interpretation

Bài TOEIC là một ví dụ đếm có điều kiện trên không gian điểm số rời rạc. Bài sản phẩm hỏng ở lượt thứ 10 lại là bài đối xứng: trước khi nhìn thứ tự, mỗi vị trí trong 10 lượt bốc đều có cùng xác suất chứa một sản phẩm hỏng, nên xác suất ở lượt thứ 10 đúng bằng tỷ lệ hàng hỏng chung là 1%.


## 5. Task 4 - Sequential Bayesian update cho xét nghiệm COVID

Task này cho thấy cập nhật Bayes không phải là một phép tính một lần rồi xong. Khi dữ liệu đến theo chuỗi, posterior sau bước trước trở thành prior cho bước sau, và toàn bộ quá trình hình thành một đường đi của niềm tin qua thời gian.

### Ý nghĩa của bài tập

Ý nghĩa lớn nhất của bài là giúp sinh viên thấy bằng chứng có thể kéo niềm tin theo nhiều hướng khác nhau. Một vài kết quả dương tính đầu có thể đẩy posterior lên rất cao, nhưng các kết quả âm tính sau đó vẫn có thể kéo nó xuống lại nếu xét nghiệm đủ đáng tin.

Bài này cũng củng cố một tính chất quan trọng của Bayes: ta có thể cập nhật tuần tự mà không cần lưu lại toàn bộ lịch sử dưới dạng công thức mới mỗi lần. Chỉ cần giữ posterior hiện tại rồi hấp thụ thêm dữ liệu mới.


### Đề bài nhắc lại

**Bài toán COVID và cập nhật tuần tự.** Xác suất mắc bệnh ban đầu là $0.1$. Xét nghiệm có sai lầm loại I là $0.02$ và loại II là $0.03$. Một người làm 10 xét nghiệm với chuỗi kết quả
$$+, +, -, +, -, -, +, -, -, -$$
Hãy thiết lập công thức và theo dõi dãy xác suất mắc bệnh của người này sau từng đợt xét nghiệm.


In [ ]:
# Theo dõi posterior qua từng lần xét nghiệm để thấy niềm tin có thể tăng rồi giảm mạnh theo chuỗi dữ liệu.
results = ['+', '+', '-', '+', '-', '-', '+', '-', '-', '-']
prior = 0.1
false_positive = 0.02
false_negative = 0.03
sensitivity = 1 - false_negative
specificity = 1 - false_positive

posterior_path = [prior]
for outcome in results:
    if outcome == '+':
        prior = sensitivity * prior / (sensitivity * prior + false_positive * (1 - prior))
    else:
        prior = false_negative * prior / (false_negative * prior + specificity * (1 - prior))
    posterior_path.append(prior)

for step, value in enumerate(posterior_path):
    label = 'prior' if step == 0 else f'{step}:{results[step-1]}'
    print(f"{label:>6} -> {value:.5f}")

plt.plot(range(len(posterior_path)), posterior_path, marker='o')
plt.xticks(range(len(posterior_path)), ['prior'] + results)
plt.ylabel('P(disease | data)')
plt.title('Posterior sau từng lần xét nghiệm')
plt.show()


## 6. Task 5 - Xạ thủ, simulation lá bài và bài toán mở rộng

Task cuối cùng gom các bài mà cấu trúc xác suất không còn đồng nhất: có bài cần tách trường hợp, có bài dùng công thức tổng quát cho nhiều xạ thủ, và có bài nên dùng simulation vì không gian mẫu quá lớn để đếm tay tiện lợi.

### Ý nghĩa của bài tập

Ý nghĩa của nhóm bài này là mở rộng “hộp đồ nghề” của người học. Không phải bài nào cũng giải bằng một công thức kinh điển duy nhất; đôi khi ta phải tổ chức lại bài toán thành các hàm xác suất nhỏ, đôi khi phải dựa vào mô phỏng để kiểm tra trực giác.

Đây là bước chuyển tốt từ xác suất lớp học sang thực hành phân tích: người giải bài phải biết chọn biểu diễn nào giúp bài toán sáng ra, chứ không chỉ thay số vào công thức có sẵn.


### Đề bài nhắc lại

Phần này gộp ba yêu cầu từ lab gốc:

- **Xạ thủ.** Cho danh sách xác suất bắn trúng $p_1,\dots,p_n$. Viết hàm tính: (a) xác suất tất cả cùng bắn trúng, biết người 1 đã trúng; (b) xác suất không ai trúng, biết có $n-1$ người nào đó đã trượt; (c) xác suất có đúng hai người trúng, biết rằng có ít nhất một người trúng.
- **Simulation bài lá.** Chia ngẫu nhiên $13$ lá trong bộ $52$ lá cho một người; ước lượng xác suất người đó tới trắng với điều kiện có 6 đôi hoặc một tứ quý Heo.
- **Bài toán mở rộng Monty Hall.** Viết công thức hoặc chương trình cho trường hợp số cửa là $n$ bất kỳ.


In [ ]:
# Gom các bài còn lại vào một cell để nhấn mạnh cách viết hàm xác suất tổng quát và một ví dụ simulation phức tạp hơn.
def calculate_prob(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    p_all_hit_given_first_hit = np.prod(probabilities[1:])
    p_none_hit_given_specific_n_minus_1_miss = 1 - probabilities[-1]
    p_exactly_two = 0.0
    n = len(probabilities)
    for i in range(n):
        for j in range(i + 1, n):
            p_exactly_two += probabilities[i] * probabilities[j] * np.prod(1 - np.delete(probabilities, [i, j]))
    p_at_least_one = 1 - np.prod(1 - probabilities)
    return p_all_hit_given_first_hit, p_none_hit_given_specific_n_minus_1_miss, p_exactly_two / p_at_least_one

example_probs = [0.8, 0.6, 0.7, 0.9]
print('Marksmen example:', calculate_prob(example_probs))

ranks = np.repeat(np.arange(13), 4)
rank_two = 0

def winning_hand(hand):
    counts = Counter(hand)
    six_pairs = sorted(counts.values()) == [1, 2, 2, 2, 2, 2, 2]
    four_twos = counts[rank_two] == 4
    return six_pairs or four_twos

N = 20000
card_sim = np.mean([winning_hand(rng.choice(ranks, size=13, replace=False)) for _ in range(N)])
print(f"Estimated P(6 pairs or four twos) by simulation = {card_sim:.6f}")


## 7. Model criticism and reflection

Simulation rất mạnh nhưng không miễn cho ta khỏi phải nghĩ về mô hình. Nếu hàm mô phỏng Monty Hall mở cửa sai quy tắc, kết quả sẽ lệch. Nếu bài sản phẩm hỏng bị hiểu nhầm là “bốc độc lập” thay vì “không hoàn lại”, xác suất cũng đổi hẳn. Vì vậy, mô phỏng tốt luôn bắt đầu từ một mô tả sinh dữ liệu chính xác.


## 8. Conceptual interpretation questions

1. Vì sao trong Monty Hall, hành vi của người dẫn chương trình là phần quan trọng nhất của mô hình?  
2. Điều gì làm bài bốc bi “có hoàn lại” và “không hoàn lại” khác nhau về mặt xác suất?  
3. Trong bài COVID, tại sao posterior có thể tăng rồi giảm mạnh theo chuỗi kết quả xét nghiệm?  
4. Khi nào simulation nên được dùng để kiểm tra một công thức giải tích?


## 9. Final takeaway

Lab 2 cho thấy công thức và simulation không đối lập nhau. Công thức cho ta cơ chế, còn simulation cho ta cách kiểm tra trực giác và nhìn rõ hậu quả của từng giả định sinh dữ liệu.


## 10. Lecture references

Nếu muốn dùng course để tự gỡ từng cụm bài trong Lab 2, nên đọc lại các lecture sau:

- [Bài 0.1: Nền tảng Xác suất - Ngôn ngữ của Sự Không chắc chắn](../../contents/vi/chapter00/_posts/2025-01-02-00_01_probability_basics.md): **Task 2** và **Task 3**: đọc lại phần tổ hợp, xác suất có điều kiện và đối xứng để phân biệt rõ bốc có hoàn lại, không hoàn lại, và các bài đếm theo vị trí.
- [Bài 1.3: Định lý Bayes - Công cụ Cập nhật Niềm tin](../../contents/vi/chapter01/_posts/2025-01-02-01_03_bayes_theorem_posterior.md): **Task 1** và **Task 4**: đọc lại cách dữ liệu mới làm đổi xác suất của trạng thái ẩn, đặc biệt hữu ích cho Monty Hall và chuỗi xét nghiệm COVID.
- [Bài 2.2: Likelihood - Câu chuyện về Dữ liệu và Mô hình](../../contents/vi/chapter02/_posts/2025-01-02-02_02_likelihood.md): **Task 1**, **Task 2** và **Task 4**: đọc lại ý “cơ chế sinh dữ liệu là một phần của mô hình” để không mô phỏng sai cách người dẫn mở cửa hay sai kiểu bốc bi.
- [Bài 2.4: Posterior - Cập nhật Niềm tin với Dữ liệu](../../contents/vi/chapter02/_posts/2025-01-02-02_04_posterior.md): **Task 4**: đọc lại cách lấy posterior sau mỗi lần quan sát và dùng posterior hiện tại làm prior cho bước kế tiếp.
- [Bài 2.6: Grid Approximation - Cầu nối từ Giải tích sang Tính toán](../../contents/vi/chapter02/_posts/2025-01-02-02_06_grid_approximation.md): **Task 1** và **Task 5**: đọc lại khi muốn hiểu simulation đang kiểm tra trực giác gì, hội tụ về đâu, và vì sao cần mô tả đúng cơ chế trước khi chạy mô phỏng.
